<h1> Earthquakes Project </h1>

In [34]:
import pandas as pd
import requests

In [35]:
# 1. Query the USGS API.
url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
params = {
    "format": "geojson",
    "eventtype": "earthquake",
    "minmagnitude": 2.0,
    "starttime": "2026-10-01",
    "endtime": "2026-10-31",
    "limit": 20000
}

In [36]:
response = requests.get(url, params=params)
response.raise_for_status()
all_earthquakes = response.json()

In [37]:
# 2. Load into a DataFrame.
df = pd.json_normalize(all_earthquakes["features"])
print(f"Target population size (N): {len(df)}")
df.head()

Target population size (N): 3747


,type,id,properties.mag,properties.place,properties.time,properties.updated,properties.tz,properties.url,properties.detail,properties.felt,...,properties.types,properties.nst,properties.dmin,properties.rms,properties.gap,properties.magType,properties.type,properties.title,geometry.type,geometry.coordinates
0,Feature,us7000tuuw,4.5,"120 km E of Port-Olry, Vanuatu",1793404592771,1794782486040,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,...,",origin,phase-data,",21,6.760,0.42,83,mb,earthquake,"M 4.5 - 120 km E of Port-Olry, Vanuatu",Point,"[168.1797, -14.8682, 10]"
1,Feature,us7000twmf,4.4,"Kermadec Islands, New Zealand",1793404454074,1794791086040,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,...,",origin,phase-data,",11,0.972,0.21,178,mb,earthquake,"M 4.4 - Kermadec Islands, New Zealand",Point,"[-177.9668, -30.2201, 10]"
2,Feature,aka2026vnovmh,2.9,"74 km SSE of McGrath, Alaska",1793401571549,1794890461040,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,...,",origin,phase-data,",37,1.000,0.60,80,ml,earthquake,"M 2.9 - 74 km SSE of McGrath, Alaska",Point,"[-154.918, 62.367, 5]"
3,Feature,us7000twmh,4.2,Fiji region,1793401112275,1794717652040,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,...,",origin,phase-data,",14,4.246,0.78,119,mb,earthquake,M 4.2 - Fiji region,Point,"[-177.7678, -19.7554, 518.336]"
4,Feature,aka2026vnojyv,3.1,"54 km WNW of Anchor Point, Alaska",1793400781297,1794890046040,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,...,",origin,phase-data,",123,0.100,0.60,57,ml,earthquake,"M 3.1 - 54 km WNW of Anchor Point, Alaska",Point,"[-152.785, 59.889, 102.5]"


In [38]:
# 3. Convenience sample vs. Simple Random Sample (n = 100)
conv_df = df.head(100).copy()
srs_df = df.sample(n=100, random_state=42)

In [39]:
# 4. Compare mean magnitude and time span against the population
def get_stats(data, name):
  times = pd.to_datetime(data["properties.time"], unit="ms")
  return {
      "Group": name,
      "Mean Magnitude": round(data["properties.mag"].mean(), 4),
      "Min Time (UTC)": times.min(),
      "Max Time (UTC)": times.max(),
      "Time Span": times.max() - times.min(),
  }


summary_df = pd.DataFrame([
    get_stats(df, "Population (mu)"),
    get_stats(conv_df, "Convenience Sample (x-bar)"),
    get_stats(srs_df, "Simple Random Sample (x-bar)"),
])
summary_df

,Group,Mean Magnitude,Min Time (UTC),Max Time (UTC),Time Span
0,Population (mu),3.3539,2026-10-01 00:00:21.945,2026-10-30 23:56:32.771,29 days 23:56:10.826000
1,Convenience Sample (x-bar),3.0239,2026-10-30 01:05:02.221,2026-10-30 23:56:32.771,0 days 22:51:30.550000
2,Simple Random Sample (x-bar),3.3859,2026-10-01 00:01:59.630,2026-10-30 21:01:33.158,29 days 20:59:33.528000


By default, the USGS API returns earthquakes sorted from newest to oldest. Taking the first 100 rows introduces selection bias because it only captures roughly the last day of October rather than spreading across the entire month. Because earthquakes cluster in time and location (such as localized swarms and aftershock sequences), sampling from a narrow time window does not represent the full population and skews the sample mean ($\bar{x}$) away from the true population mean ($\mu$).

In [40]:
# 1. Extract the region label from the JSON data.
regions = []
for quake in all_earthquakes["features"]:
  place = quake["properties"]["place"]
  if place:
    regions.append(place.split(", ")[-1])
  else:
    regions.append("Unknown")

df["region"] = regions


# 2. Group regions into 4 broad strata.
def get_stratum(region):
  north_america = [
      "Alaska",
      "CA",
      "California",
      "Hawaii",
      "Nevada",
      "Texas",
      "Oklahoma",
      "Washington",
      "Oregon",
      "Utah",
      "Idaho",
      "Montana",
      "Puerto Rico",
      "Mexico",
      "Canada",
  ]
  asia_pacific = [
      "Japan",
      "Indonesia",
      "Philippines",
      "Papua New Guinea",
      "Vanuatu",
      "Fiji",
      "Fiji region",
      "Tonga",
      "New Zealand",
      "Solomon Islands",
      "Russia",
      "Taiwan",
      "China",
      "India",
      "Iran",
      "Turkey",
  ]
  south_america = [
      "Chile",
      "Peru",
      "Argentina",
      "Ecuador",
      "Colombia",
      "Bolivia",
  ]

  if region in north_america:
    return "North America"
  elif region in asia_pacific:
    return "Asia & Pacific"
  elif region in south_america:
    return "South America"
  else:
    return "Europe, Africa & Oceans"


df["stratum"] = df["region"].apply(get_stratum)

# Check the counts and preview the new columns.
print(df["stratum"].value_counts())
df[["properties.place", "region", "stratum"]].head()

stratum
North America              1999
Asia & Pacific              867
Europe, Africa & Oceans     787
South America                94
Name: count, dtype: int64


,properties.place,region,stratum
0,"120 km E of Port-Olry, Vanuatu",Vanuatu,Asia & Pacific
1,"Kermadec Islands, New Zealand",New Zealand,Asia & Pacific
2,"74 km SSE of McGrath, Alaska",Alaska,North America
3,Fiji region,Fiji region,Asia & Pacific
4,"54 km WNW of Anchor Point, Alaska",Alaska,North America


In [41]:
# 1. Find each stratum's share of the population and multiply by 200.
n_total = 200
shares = df["stratum"].value_counts(normalize=True)
sample_sizes = (shares * n_total).round().astype(int)

# Make sure rounding still adds up to exactly 200 rows.
sample_sizes.iloc[0] += n_total - sample_sizes.sum()

# 2. Draw that exact number of random rows from each stratum.
samples = []
for stratum_name, count in sample_sizes.items():
  group = df[df["stratum"] == stratum_name]
  samples.append(group.sample(n=count, random_state=42))

# 3. Combine the 4 mini-samples into one DataFrame.
strat_df = pd.concat(samples)

# 4. Check that the total is 200 and see the counts per group.
print(f"Total stratified sample size: {len(strat_df)}")
print(strat_df["stratum"].value_counts())

Total stratified sample size: 200
stratum
North America              107
Asia & Pacific              46
Europe, Africa & Oceans     42
South America                5
Name: count, dtype: int64


In [42]:
# Variance of the Simple Random Sample's mean: s^2 / n
srs_var_mean = srs_df["properties.mag"].var() / len(srs_df)

# Variance of the Stratified Sample's mean: sum(W_h^2 * s_h^2 / n_h)
N = len(df)
strat_var_mean = 0
for stratum_name, group in strat_df.groupby("stratum"):
  W_h = (df["stratum"] == stratum_name).sum() / N
  s_h_sq = group["properties.mag"].var()
  n_h = len(group)
  strat_var_mean += (W_h**2) * (s_h_sq / n_h)

print(f"SRS Mean Magnitude: {srs_df['properties.mag'].mean():.4f}")
print(f"Stratified Mean Magnitude: {strat_df['properties.mag'].mean():.4f}")
print(f"SRS Variance of Sample Mean: {srs_var_mean:.6f}")
print(f"Stratified Variance of Sample Mean: {strat_var_mean:.6f}")

SRS Mean Magnitude: 3.3859
Stratified Mean Magnitude: 3.3575
SRS Variance of Sample Mean: 0.014748
Stratified Variance of Sample Mean: 0.001905


### Comparison of Sample Mean Variance: Stratified Sample vs. Simple Random Sample

The **stratified sample's mean magnitude has a noticeably lower variance** than the simple random sample's mean magnitude for two reasons:

1. **Larger Sample Size ($n = 200$ vs. $n = 100$):** Because the variance of a sample mean ($\text{Var}(\bar{x}) = \frac{s^2}{n}$) shrinks as sample size grows, doubling the sample size from 100 to 200 cuts the sampling variance roughly in half.
2. **Elimination of Between-Region Variation:** Different geographic regions experience systematically different typical earthquake magnitudes (for example, North America has a high volume of smaller crustal earthquakes, whereas the Asia & Pacific region has a higher share of larger subduction-zone earthquakes). Because proportional stratification guarantees that every region is represented in its exact population share, variation *between* strata is removed, leaving only within-stratum variance and producing a more precise, stable estimate of the true population mean ($\mu$).